In [6]:
import numpy as np
import time
import matplotlib.pyplot as plt
import time
import scipy.io as sio
import sys

In [7]:
def means_and_cov(d,i_group):
    
    S = np.loadtxt("Data_projeto/SpikingPopulation_"+str(String[d])+"_"+str(i_group)+".txt")
    
    S=np.where(S==0,-1,S)
    means=np.mean(S,axis=1)   
    cov=np.cov(S)    
    S=np.where(S==-1,0,S)
    
    return means,cov,S

In [8]:
def Exact_mc(N,h,J):
    state=np.zeros(N,dtype=int)   # N number of neurons
    means_=np.zeros(N)
    state_mean=np.zeros(N)
    cov_=np.zeros((N,N))
    cov=np.zeros((N,N))
    N_states=2**N

    Z=0
    for st in range(N_states):
        bina=np.binary_repr(st,N)
        #print(bina)
        for i in range(N):
            state[i]=-1+2*int(bina[i])
        #print(state)
        suma=0
        for i in range(N-1):
            suma+=np.sum((state[i]*state[i+1:])*J[i,i+1:])
        p_=np.exp(np.sum(h*state)+suma)
        Z+=p_

        means_+=state*p_
        for i in range(N-1):
            cov_[i,i+1:]+=(state[i]*state[i+1:])*p_
    
    means=means_/Z
    cov=cov_/Z
    for i in range(N):
        cov[i,:]=cov[i,:]-means[i]*means
    cov=cov-np.tril(cov)
    np.fill_diagonal(cov,0)
    return (means,cov)

In [9]:
def GD_mc(h,J,means,cov,means_obs,cov_obs,ite,means_er,cov_er,ex,alpha,beta):
     
    a_h=0
    a_j=0
    a=0
    err_j=np.zeros((N,N))    
    err_h=np.abs((means_obs-means)/means_obs)
    
    for i in range(N):
        err_j[i,i+1:]=np.abs((cov_obs[i,i+1:]-cov[i,i+1:])/cov_obs[i,i+1:])

        if (err_h[i]>means_er):
            h[i]=h[i]+alpha*(means_obs[i]-means[i])
        else:
            a_h+=1
        for j in range(1,N-i):
            if (err_j[i,j+i]<cov_er):
                a_j+=1
            else:
                J[i,j+i]=J[i,j+i]+beta*(cov_obs[i,j+i]-cov[i,j+i])

    if a_h==N:
        a+=1
    if a_j==int((N*N-N)/2):
        a+=1
    if a==2:
        print("Done_all",ite)
    #else:
        #print("a_h, a_j",np.array([a_h,a_j]))
    return (h,J,a)

In [ ]:
############################### Main ########################
neurona=0
d=1     # 0 If hippocampus, 1 if retina
String = ['Hippocampus', 'Retina']

means_,cov_,S=means_and_cov(d,neurona)   #This is already organized by mutual information with the chosen neuron. 
N_time=S.shape[1]
N_neurons = S.shape[0]
print(N_neurons)

N_array = np.array([3, 5, 7])

for grupo in range(3):
    print("grupo", grupo)
    t0 = time.time()
    N = N_array[grupo]
    
    means_obs=np.zeros(N)
    cov_obs=np.zeros((N,N))
    cov_obs_=np.zeros((N,cov_.shape[0]))

    idx_neurons = np.random.choice(N_neurons, N, replace = False) 
    means_obs=means_[idx_neurons]
    cov_obs_=cov_[idx_neurons,:]
    cov_obs=cov_obs_[:, idx_neurons]
    cov_obs=cov_obs-np.tril(cov_obs)

    h=np.zeros(N)
    J=np.zeros((N,N))     # We will keep this with zeros for the independent model. 

    ite=100000
    contador=0
    fin_ite=ite
    if d==1:
        alpha=0.7    #For Retina
        beta=0.4     
    else:
        alpha=1    #For hippo
        beta=1     

    for iteration in range(ite):
        #print("iteration_",iteration)
        means,cov=Exact_mc(N,h,J)
        h,J,a=GD_mc(h,J,means,cov,means_obs,cov_obs,iteration,0.001,0.001,0,alpha,beta)
        if a>1:
            fin_ite=iteration
            print("Learned in time", time.time() - t0)
            break
    
    plt.plot(means_obs, means, marker = '.', linestyle = 'none')
    plt.xlabel("Model means", fontsize = 12)
    plt.xlabel("Experimental means", fontsize = 12)
    plt.show()
    plt.plot(np.reshape(cov_obs, N*N), np.reshape(cov, N*N), marker = '.', linestyle = 'none')
    plt.xlabel("Experimental cov", fontsize = 12)
    plt.show()